# 05 - Lexicon baseline (RQ1)

A frozen, theory-derived rule-based lexicon for the three constructs:
grievance (1A), weaponised victimhood (1B = 1A + a "move"), and conspiratorial
framing (2 = coalition + asserted agency). Terms are grouped by construct and
matched on word boundaries after normalisation.

Developed and evaluated on the training split (`train_fit.csv`); the held-out
test set is scored once, separately, only after this lexicon is frozen.

In [ ]:
import re
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, precision_recall_fscore_support

# Resolve the repo root whether the notebook runs from notebooks/ or the root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
proc = PROJECT_ROOT / "data" / "processed"

# The lexicon is rule-based (not trained), so it is developed on the FULL
# development set = train_fit + validation (1620 rows). The test split (406)
# stays held out and is scored once, later, only after the lexicon is frozen.
train_fit = pd.read_csv(proc / "train_fit.csv")
validation = pd.read_csv(proc / "validation.csv")
df = pd.concat([train_fit, validation], ignore_index=True)

label_cols = ["label_1A", "label_1B", "label_2"]
text_col = "text_clean"
for c in label_cols:
    df[c] = pd.to_numeric(df[c], errors="raise").astype(int)

In [ ]:
# Theory-derived term lists, grouped by construct. Each group is one "signal";
# the predictors below combine signals into a construct decision. Terms are
# matched on word boundaries after normalisation (see next cell).

# ---- 1A: male in-group + harm ----
# A grievance needs BOTH a male in-group reference AND a harm/injustice claim,
# unless a "strong" standalone phrase already implies both.
ingroup_terms = ["men","man","male","males","boys","boy","fathers","father","dads","dad",
    "husbands","husband","sons","son","young men","white men","western men","brothers",
    "gentlemen","guys","male experience","men's issues","mens issues","men's rights","mens rights",
    "we","we're","we are"]

# Harm / injustice / victimhood done to that in-group.
harm_terms = ["under attack","attacked","attacking","attack","targeted","oppressed","silenced",
    "censored","suppressed","ignored","abandoned","mocked","ridiculed","demonized","demonised",
    "vilified","hated","blamed","punished","discriminated","excluded","marginalized","marginalised",
    "disadvantaged","exploited","used","abused","destroyed","ruined","emasculated","cancelled",
    "canceled","fired","blacklisted","disposable","screwed","biased","rigged","stacked against",
    "unfair","double standard","double standards","no accountability","get away with",
    "false accusation","false accusations","false allegation","false allegations","falsely accused",
    "lied about","no rights","no voice","no respect","nothing to lose","nothing left","no future",
    "get nothing","gets nothing","lose everything","second class","treated like slaves",
    "lose the house","lose your house","take your house","lose the kids","lose your kids",
    "take your kids","lose custody","child support","alimony","take all your money",
    "financially ruined","bankrupted","forced to","expected to","have to pay","have to provide",
    "have to protect","have to sacrifice","die for nothing","pay for nothing","work for nothing",
    "nobody cares","no one cares","loneliness","suicide","system's against","system is against",
    "subjected to","subject to","prey","predatory","enslaved","expendable","deceived","lied to",
    "sacrificed","taken for granted","set up to fail","thrown under the bus"]

# Standalone phrases that already assert male grievance on their own (fire 1A alone).
strong_1a_phrases = ["war on men","attack on men","anti male","anti-male","misandry","misandrist",
    "male hatred","men are under attack","men are being attacked","system is against men",
    "system against men","stacked against men","rigged against men","biased against men",
    "men have no rights","fathers have no rights","men are disposable","men are the real victims",
    "nobody cares about men","no one cares about men","men get blamed","men are blamed",
    "men get punished","men are punished","men get nothing","men lose everything",
    "men are being silenced","men are silenced","men are ignored","men are oppressed",
    "men are discriminated against","male loneliness","male suicide","society hates men",
    "men are guilty until proven innocent","marriage is a bad deal","marriage is obsolete",
    "don't get married","do not get married","never get married","family court is broken",
    "family courts are broken","biased courts","court favours women","court favors women",
    "courts favour women","courts favor women","we are being silenced","we are being attacked",
    "we are the victims","men are prey","men are the prey","men are expendable",
    "men are treated like slaves","men are treated as slaves","men are second class citizens",
    "men are being lied to","men have been lied to","men are set up to fail"]

# Relationship/marriage grievance: a relationship context PLUS a risk/loss framing.
relationship_terms = ["marriage","married","marry","divorce","dating","relationship","wife","wives",
    "girlfriend","family court","child support","alimony","custody","prenup"]
relationship_risk_terms = ["bad deal","obsolete","risk","lose everything","take half",
    "financially ruined","bankrupt","false accusation","false allegations","nothing to gain",
    "no benefit","no point","biased court","biased courts","lose custody","lost custody",
    "silver bullet"]
# Explicit double-standard claims also count as grievance on their own.
double_standard_terms = ["double standard","double standards","men have to","men must",
    "men are expected to","women get away with","women don't have to","women do not have to",
    "different rules"]

# ---- 1B: the "move" (weaponisation) ----
# 1B = a 1A grievance PLUS one of these moves: blame, withdrawal, retaliation,
# offensive demonising, or moral inversion. Strong 1B phrases fire on their own.
blame_terms = ["women are to blame","feminism is to blame","feminists are to blame","women caused this",
    "feminism caused this","feminists caused this","women created this","feminism created this",
    "women destroyed","feminism destroyed","women ruined","feminism ruined","they did this to men",
    "women hate us","women use us","women lie about men","feminists hate men","the system hates men",
    "hold women accountable","zero accountability","scot free","nazi feminism"]
withdrawal_terms = ["go your own way","mgtow","stay single","avoid women","avoid dating","stop dating",
    "stop marrying","don't marry","do not marry","never marry","walk away","keep your money",
    "stop paying","stop working","stop protecting","stop sacrificing","stop supporting",
    "stop hiring women","don't hire women","boycott","go on strike","men on strike"]
retaliation_terms = ["fight back","stand against","stand our ground","rise up","revolt","take back",
    "regain our rights","remove their rights","strip women","stop them from voting","remove the vote",
    "deport","military takeover","military take over","civil war","punish them","make women pay",
    "return to patriarchy","enough is enough"]
offensive_demonise_terms = ["believe all women","false accusation","false accusations",
    "false allegation","false allegations","false rape","falsely accused","fake victim","fake victims",
    "playing the victim","cry rape","weaponize","weaponise","metoo","me too","women lie","feminazi",
    "feminists are hitler","war on men","war against men","nazi feminism","genocide against men",
    "male genocide","gynocide"]
strong_1b_phrases = ["men are fighting back","regain our rights and dignity","if men went on strike",
    "men stop paying taxes","hold women accountable","women created this problem",
    "feminism created this problem","strip women of","stop women from voting","remove their rights",
    "go your own way","fight back","stand our ground","civil war","military takeover"]
# Moral inversion: recasting the out-group as undeserving of sympathy.
moral_inversion_terms = ["men are the real victims","men are the true victims","we are the noble gender",
    "men are the noble gender","can't blame men","cannot blame men","can not blame men","no wonder men",
    "what do you expect men to do","women deserve no sympathy","they deserve no sympathy",
    "not our problem","let them suffer","let women suffer","let them fend for themselves",
    "women can fend for themselves","deserve what they get","reap what they sow","let it burn",
    "collapse soon enough"]

# ---- 2: conspiratorial framing (coalition + asserted agency) ----
# 2 needs a named actor/coalition AND a mechanism of deliberate, hidden action
# (an actor + mechanism appearing close together), or a strong standalone phrase.
actor_terms = ["the state","mainstream media","the media","elites","the elites","oligarchs",
    "globalists","united nations","wef","cia","splc","bankers","the fed","central bank","feminism",
    "feminists","sisterhood","powers that be","the matrix","satanic","new world order","deep state",
    "illuminati","freemasons","the government","left wing","left-wing","liberals","democrats","ngos",
    "planned parenthood"]
mechanism_terms = ["by design","designed to","planned to","a plan to","part of the plan",
    "hidden agenda","the agenda","social engineering","engineered","deliberately","intentionally",
    "covertly","in the shadows","behind the scenes","cover up","covering up","covered up","conceal",
    "concealing","hide the truth","hiding the truth","funded","funding","paid to",
    "controlled opposition","gatekeepers","indoctrinate","indoctrinated","indoctrination","subvert",
    "subverted","orchestrated","manufactured","trying to destroy","trying to suppress","propaganda",
    "push the narrative","pushes the narrative","pushing the narrative","programmed","programming",
    "conditioned","conditioning","scheme","coordinated","coordination","collude","colluded","collusion",
    "set up to","created to","meant to","the goal is","the purpose is","suppress the evidence",
    "suppressing evidence","hide the evidence","hiding evidence","hide the statistics","hiding statistics"]
strong_2_phrases = ["by design","it was all a plan","part of the plan","social engineering",
    "in the shadows","behind the scenes","cover up","covering up","covered up","hidden agenda",
    "powers that be","the matrix","system is satanic","controlled opposition","gatekeepers",
    "systematically destroying","funding feminism"]

In [ ]:
# Matching helpers

def normalize(text):
    """Lowercase, strip URLs/@mentions, and reduce to plain words so term
    matching is consistent (curly quotes → straight, punctuation → spaces)."""
    if pd.isna(text): return ""
    t = str(text).lower().replace("’","'").replace("‘","'")
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)   # drop links
    t = re.sub(r"@\w+", " ", t)                     # drop @mentions
    t = re.sub(r"[^a-z0-9':]+", " ", t)             # keep letters/digits/'/:
    return re.sub(r"\s+", " ", t).strip()

def _compile(terms):
    """Build one regex that matches any term in the list on word boundaries,
    so 'men' does not match inside 'women', etc."""
    return re.compile(r"(?<!\w)(?:" + "|".join(re.escape(normalize(x)) for x in terms) + r")(?!\w)")

# Pre-compile every term group once (name -> compiled pattern) for speed.
PAT = {name: _compile(terms) for name, terms in dict(
    ingroup=ingroup_terms, harm=harm_terms, s1a=strong_1a_phrases,
    relctx=relationship_terms, relrisk=relationship_risk_terms, ds=double_standard_terms,
    blame=blame_terms, withdrawal=withdrawal_terms, retal=retaliation_terms,
    offdem=offensive_demonise_terms, s1b=strong_1b_phrases, moral_inv=moral_inversion_terms,
    actor=actor_terms, mech=mechanism_terms, s2=strong_2_phrases,
).items()}

# "they + deliberate action" — catches conspiratorial agency without a named actor.
THEY_ACTION = re.compile(
    r"\bthey\b.{0,150}\b(by design|the plan|the agenda|engineered|social engineering|"
    r"depopulat|the goal is|created to|funded feminism|cover up|brainwash|indoctrinat)\b")

def has(name, n):
    """True if the given term group appears in normalised text n."""
    return bool(PAT[name].search(n))

def near(a, b, n, max_chars=270):
    """True if a term from group a and a term from group b occur within
    max_chars of each other — used to require actor + mechanism co-occurrence."""
    xa = [m.start() for m in PAT[a].finditer(n)]
    xb = [m.start() for m in PAT[b].finditer(n)]
    return any(abs(i - j) <= max_chars for i in xa for j in xb)

def is_chapter_list(n):
    """Detect video-chapter/timestamp dumps (>=5 'm:ss' stamps) so long
    description pastes aren't mistaken for grievance/conspiracy text."""
    return len(re.findall(r"\b\d{1,2}:\d{2}\b", n)) >= 5

In [ ]:
# Predictors

def predict_1a(n):
    # Grievance: a strong standalone phrase, OR in-group + harm, OR a
    # relationship context + risk framing, OR an explicit double standard.
    if is_chapter_list(n): return 0                 # skip timestamp dumps
    return int(has("s1a", n)
               or (has("ingroup", n) and has("harm", n))
               or (has("relctx", n) and has("relrisk", n))
               or has("ds", n))

def predict_1b(n):
    # Weaponised victimhood: a 1A grievance PLUS a "move"
    # (blame / withdrawal / retaliation / demonising / moral inversion),
    # OR a strong standalone 1B phrase.
    if is_chapter_list(n): return 0
    move = (has("blame", n) or has("withdrawal", n) or has("retal", n)
            or has("offdem", n) or has("moral_inv", n))
    return int(has("s1b", n) or (predict_1a(n) == 1 and move))

def predict_2(n):
    # Conspiratorial framing: a strong phrase, OR actor + mechanism close
    # together, OR the 'they + deliberate action' pattern.
    if is_chapter_list(n): return 0
    return int(has("s2", n) or near("actor", "mech", n) or bool(THEY_ACTION.search(n)))

def predict_all(text):
    # Run all three on one comment. 1B logically entails 1A, so if 1B fires we
    # force 1A on too (keeps the labels internally consistent).
    n = normalize(text)
    p1b = predict_1b(n)
    p1a = max(predict_1a(n), p1b)
    return {"lexicon_1A": p1a, "lexicon_1B": p1b, "lexicon_2": predict_2(n)}

In [ ]:
# Evaluate and save

# Predict all three constructs for every training comment.
preds = df[text_col].fillna("").apply(predict_all).apply(pd.Series)
results = pd.concat([df.reset_index(drop=True), preds], axis=1)

# Sanity checks: 1B never fires without 1A, and predictions are strictly 0/1.
assert not ((results.lexicon_1B == 1) & (results.lexicon_1A == 0)).any()
for c in ["lexicon_1A", "lexicon_1B", "lexicon_2"]:
    assert set(results[c].unique()).issubset({0, 1})

# Compare lexicon predictions against the gold human labels, construct by construct.
pairs = {"label_1A": "lexicon_1A", "label_1B": "lexicon_1B", "label_2": "lexicon_2"}
for gold, pred in pairs.items():
    print("\n" + "=" * 50 + f"\n{gold}\n" + "=" * 50)
    print(classification_report(results[gold], results[pred], digits=3, zero_division=0))

# Collect precision/recall/F1 for the positive class into one tidy table.
rows = []
for gold, pred in pairs.items():
    p, r, f, s = precision_recall_fscore_support(results[gold], results[pred],
                                                 labels=[1], average=None, zero_division=0)
    rows.append({"construct": gold.replace("label_", ""), "precision": p[0],
                 "recall": r[0], "f1": f[0], "support": s[0]})
metrics = pd.DataFrame(rows).round(3)
print("\n", metrics.to_string(index=False))

# Save per-comment predictions and the summary metrics for downstream comparison.
results.to_csv(proc / "lexicon_development_predictions.csv", index=False)
metrics.to_csv(proc / "lexicon_development_metrics.csv", index=False)
print("\nSaved development predictions and metrics.")